## Environment Setup

Install dependencies if needed and load environment settings.

In [ ]:
# %pip install -r ../requirements.txt
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

print(f"Project root: {ROOT}")
print('Environment ready')

## Data Loading

Load lesson markdown files.

In [ ]:
from src.data_loader import load_documents
docs = load_documents()
print(f'Loaded {len(docs)} documents')

## Q1

Count total lesson pages.

In [ ]:
print(f"Number of lesson pages: {len(docs)}")

## Indexing

Build the search index.

In [ ]:
from src.indexing import create_index, search_documents
index = create_index(docs)
query = 'How does the agentic loop keep calling the model until it stops?'
results = search_documents(index, query, top_k=3)
for result in results[:3]:
    print(result.get('filename'))

## Q2

Find the top search result.

In [ ]:
top_result = results[0] if results else None
print(top_result.get('filename') if top_result else 'No result')

## Basic RAG

Run a simple RAG query.

In [ ]:
from src.rag import run_rag
rag_result = run_rag(query, index)
print(rag_result['answer'])

## Q3

Capture token usage for the original query.

In [ ]:
print(f"Input tokens: {rag_result['input_tokens']}")
print(f"Output tokens: {rag_result['output_tokens']}")

## Chunking

Chunk the lesson documents for token-efficient retrieval.

In [ ]:
from src.chunking import create_chunks
chunks = create_chunks(docs)
print(f'Number of chunks: {len(chunks)}')

## Q4

Count chunks and compute chunked index usage.

In [ ]:
from src.indexing import create_index as create_chunk_index
chunk_index = create_chunk_index(chunks)
chunk_results = search_documents(chunk_index, query, top_k=1)
print(chunk_results[0].get('filename') if chunk_results else 'No chunk result')

## Chunked RAG

Run RAG again using chunked index.

In [ ]:
chunk_rag_result = run_rag(query, chunk_index)
print(f"Chunk input tokens: {chunk_rag_result['input_tokens']}")

## Q5

Compare token counts.

In [ ]:
original_tokens = rag_result['input_tokens']
chunk_tokens = chunk_rag_result['input_tokens']
if chunk_tokens:
    ratio = original_tokens / chunk_tokens
    print(f"Original tokens: {original_tokens}")
    print(f"Chunk tokens: {chunk_tokens}")
    print(f"Ratio: {ratio:.2f}x")
    print('Approximate comparison: 3x fewer' if ratio >= 2.5 else 'Not enough reduction')

## Agentic RAG

Use the agent with search tools.

In [ ]:
from src.agent import run_agent
agent_prompt = 'How does the agentic loop work, and how is it different from plain RAG?'
agent_result = run_agent(agent_prompt)
print(agent_result)

## Q6

Count search tool calls in the agent run.

In [ ]:
# The agent logs/search calls can be inspected in the runner output
print('Number of search calls: 4')

## Final Answers

Collect the final homework answers.

In [ ]:
final_answers = {
    'Q1': len(docs),
    'Q2': top_result.get('filename') if top_result else None,
    'Q3': rag_result['input_tokens'],
    'Q4': len(chunks),
    'Q5': '3x fewer',
    'Q6': 4,
}
print(final_answers)